In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from config import (
    RAW_DATA_PATH,
    PROCESSED_DATA_PATH,
    MIN_PRICE,
    MIN_AVG_DOLLAR_VOLUME,
)


In [8]:
!which python

/opt/homebrew/Caskroom/miniconda/base/bin/python


In [9]:
def load_raw_data():
    path = Path(RAW_DATA_PATH) / "us_equities_ohlcv.parquet"
    df = pd.read_parquet(path)
    return df

df_us = load_raw_data()

df_us.head()

,"('date', '')","('adj_close', 'AAPL')","('close', 'AAPL')","('high', 'AAPL')","('low', 'AAPL')","('open', 'AAPL')","('volume', 'AAPL')","('ticker', '')","('adj_close', 'MSFT')","('close', 'MSFT')",...,"('high', 'NKE')","('low', 'NKE')","('open', 'NKE')","('volume', 'NKE')","('adj_close', 'SBUX')","('close', 'SBUX')","('high', 'SBUX')","('low', 'SBUX')","('open', 'SBUX')","('volume', 'SBUX')"
0,2010-01-04,6.418382,7.643214,7.660714,7.585000,7.622500,493729600.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-05,6.429479,7.656429,7.699643,7.616071,7.664286,601904800.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-06,6.327210,7.534643,7.686786,7.526786,7.656429,552160000.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-01-07,6.315512,7.520714,7.571429,7.466071,7.562500,477131200.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-01-08,6.357501,7.570714,7.571429,7.466429,7.510714,447610800.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df_us.columns

Index(['('date', '')', '('adj_close', 'AAPL')', '('close', 'AAPL')',
       '('high', 'AAPL')', '('low', 'AAPL')', '('open', 'AAPL')',
       '('volume', 'AAPL')', '('ticker', '')', '('adj_close', 'MSFT')',
       '('close', 'MSFT')',
       ...
       '('high', 'NKE')', '('low', 'NKE')', '('open', 'NKE')',
       '('volume', 'NKE')', '('adj_close', 'SBUX')', '('close', 'SBUX')',
       '('high', 'SBUX')', '('low', 'SBUX')', '('open', 'SBUX')',
       '('volume', 'SBUX')'],
      dtype='object', length=242)

In [11]:
import ast

def restore_multicolumns(df):
    new_cols = []

    for col in df.columns:
        if col.startswith("(") and col.endswith(")"):
            new_cols.append(ast.literal_eval(col))
        else:
            # Handles columns like ('date','') or malformed leftovers
            new_cols.append((col, ""))

    df.columns = pd.MultiIndex.from_tuples(new_cols)
    return df

def normalize_price_fields(df):
    df = df.copy()
    df.columns = pd.MultiIndex.from_tuples([
        (field.lower().replace(" ", "_"), ticker)
        for field, ticker in df.columns
    ])
    return df

def wide_to_long_panel(df):
    # Move date to index if needed
    if ("date", "") in df.columns:
        df = df.set_index(("date", ""))

    df.index.name = "date"

    # Stack ticker dimension
    df = df.stack(level=1).reset_index()

    # Rename columns
    df = df.rename(columns={"level_1": "ticker"})

    return df

df = df_us.copy()

df = restore_multicolumns(df)
df = normalize_price_fields(df)
df = wide_to_long_panel(df)

df.head()

/var/folders/gx/p5qhhh8s7rg2_ywh443qwth80000gn/T/ipykernel_6559/3142940936.py:32: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(level=1).reset_index()


,date,ticker,adj_close,close,high,low,open,volume,ticker
0,2010-01-04,,NaN,NaN,NaN,NaN,NaN,NaN,AAPL
1,2010-01-04,AAPL,6.418382,7.643214,7.660714,7.585000,7.622500,493729600.0,NaN
2,2010-01-05,,NaN,NaN,NaN,NaN,NaN,NaN,AAPL
3,2010-01-05,AAPL,6.429479,7.656429,7.699643,7.616071,7.664286,601904800.0,NaN
4,2010-01-06,,NaN,NaN,NaN,NaN,NaN,NaN,AAPL


In [12]:
df

,date,ticker,adj_close,close,high,low,open,volume,ticker
0,2010-01-04,,NaN,NaN,NaN,NaN,NaN,NaN,AAPL
1,2010-01-04,AAPL,6.418382,7.643214,7.660714,7.585000,7.622500,493729600.0,NaN
2,2010-01-05,,NaN,NaN,NaN,NaN,NaN,NaN,AAPL
3,2010-01-05,AAPL,6.429479,7.656429,7.699643,7.616071,7.664286,601904800.0,NaN
4,2010-01-06,,NaN,NaN,NaN,NaN,NaN,NaN,AAPL
...,...,...,...,...,...,...,...,...,...
298885,2024-12-26,XOM,102.720039,106.489998,107.029999,105.940002,106.519997,9652400.0,NaN
298886,2024-12-27,,NaN,NaN,NaN,NaN,NaN,NaN,XOM
298887,2024-12-27,XOM,102.710388,106.480003,107.989998,105.769997,106.300003,11943900.0,NaN
298888,2024-12-30,,NaN,NaN,NaN,NaN,NaN,NaN,XOM


In [13]:
df.columns

Index(['date', 'ticker', 'adj_close', 'close', 'high', 'low', 'open', 'volume',
       'ticker'],
      dtype='object')

In [14]:
if list(df.columns).count("ticker") > 1:
    df = df.loc[:, ~df.columns.duplicated()]

In [15]:
df

,date,ticker,adj_close,close,high,low,open,volume
0,2010-01-04,,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-04,AAPL,6.418382,7.643214,7.660714,7.585000,7.622500,493729600.0
2,2010-01-05,,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-01-05,AAPL,6.429479,7.656429,7.699643,7.616071,7.664286,601904800.0
4,2010-01-06,,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
298885,2024-12-26,XOM,102.720039,106.489998,107.029999,105.940002,106.519997,9652400.0
298886,2024-12-27,,NaN,NaN,NaN,NaN,NaN,NaN
298887,2024-12-27,XOM,102.710388,106.480003,107.989998,105.769997,106.300003,11943900.0
298888,2024-12-30,,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
def basic_cleaning(df):
    df = df.copy()

    df = df[df["close"] > 0]
    df = df[df["volume"] > 0]

    df = df.sort_values(["ticker", "date"])
    df.reset_index(drop=True, inplace=True)

    return df

df_us2 = basic_cleaning(df)

df_us2.head()

,date,ticker,adj_close,close,high,low,open,volume
0,2010-01-04,AAPL,6.418382,7.643214,7.660714,7.585000,7.622500,493729600.0
1,2010-01-05,AAPL,6.429479,7.656429,7.699643,7.616071,7.664286,601904800.0
2,2010-01-06,AAPL,6.327210,7.534643,7.686786,7.526786,7.656429,552160000.0
3,2010-01-07,AAPL,6.315512,7.520714,7.571429,7.466071,7.562500,477131200.0
4,2010-01-08,AAPL,6.357501,7.570714,7.571429,7.466429,7.510714,447610800.0


In [17]:
df_us2

,date,ticker,adj_close,close,high,low,open,volume
0,2010-01-04,AAPL,6.418382,7.643214,7.660714,7.585000,7.622500,493729600.0
1,2010-01-05,AAPL,6.429479,7.656429,7.699643,7.616071,7.664286,601904800.0
2,2010-01-06,AAPL,6.327210,7.534643,7.686786,7.526786,7.656429,552160000.0
3,2010-01-07,AAPL,6.315512,7.520714,7.571429,7.466071,7.562500,477131200.0
4,2010-01-08,AAPL,6.357501,7.570714,7.571429,7.466429,7.510714,447610800.0
...,...,...,...,...,...,...,...,...
149439,2024-12-23,XOM,102.536774,106.300003,106.599998,104.919998,105.309998,12285100.0
149440,2024-12-24,XOM,102.633224,106.400002,107.190002,105.699997,106.519997,7807000.0
149441,2024-12-26,XOM,102.720039,106.489998,107.029999,105.940002,106.519997,9652400.0
149442,2024-12-27,XOM,102.710388,106.480003,107.989998,105.769997,106.300003,11943900.0


In [18]:
df_us2['ticker'].nunique()

40

In [19]:
#Now checking if the stocks are liquid enough to be considered

def liquidity_filter(df):
    
    df = df.copy()
    
    df["dollar_volume"] = df["close"]*df["volume"]
    
    liquidity = (df.groupby("ticker")['dollar_volume'].mean().rename("avg_dollar_volume)"))
    
    liquid_tickers = liquidity[liquidity>= MIN_AVG_DOLLAR_VOLUME].index
    
    df[df['ticker'].isin(liquid_tickers)]
    
    return df

df = liquidity_filter(df_us2)

df

,date,ticker,adj_close,close,high,low,open,volume,dollar_volume
0,2010-01-04,AAPL,6.418382,7.643214,7.660714,7.585000,7.622500,493729600.0,3.773681e+09
1,2010-01-05,AAPL,6.429479,7.656429,7.699643,7.616071,7.664286,601904800.0,4.608441e+09
2,2010-01-06,AAPL,6.327210,7.534643,7.686786,7.526786,7.656429,552160000.0,4.160329e+09
3,2010-01-07,AAPL,6.315512,7.520714,7.571429,7.466071,7.562500,477131200.0,3.588367e+09
4,2010-01-08,AAPL,6.357501,7.570714,7.571429,7.466429,7.510714,447610800.0,3.388733e+09
...,...,...,...,...,...,...,...,...,...
149439,2024-12-23,XOM,102.536774,106.300003,106.599998,104.919998,105.309998,12285100.0,1.305906e+09
149440,2024-12-24,XOM,102.633224,106.400002,107.190002,105.699997,106.519997,7807000.0,8.306648e+08
149441,2024-12-26,XOM,102.720039,106.489998,107.029999,105.940002,106.519997,9652400.0,1.027884e+09
149442,2024-12-27,XOM,102.710388,106.480003,107.989998,105.769997,106.300003,11943900.0,1.271787e+09


In [20]:
#Now the price filter 

def apply_price_filter(df):
    df = df.copy()

    avg_price = (
        df.groupby("ticker")["close"]
        .mean()
        .rename("avg_price")
    )

    valid_tickers = avg_price[avg_price >= MIN_PRICE].index
    df = df[df["ticker"].isin(valid_tickers)]

    return df

df = apply_price_filter(df)

df

,date,ticker,adj_close,close,high,low,open,volume,dollar_volume
0,2010-01-04,AAPL,6.418382,7.643214,7.660714,7.585000,7.622500,493729600.0,3.773681e+09
1,2010-01-05,AAPL,6.429479,7.656429,7.699643,7.616071,7.664286,601904800.0,4.608441e+09
2,2010-01-06,AAPL,6.327210,7.534643,7.686786,7.526786,7.656429,552160000.0,4.160329e+09
3,2010-01-07,AAPL,6.315512,7.520714,7.571429,7.466071,7.562500,477131200.0,3.588367e+09
4,2010-01-08,AAPL,6.357501,7.570714,7.571429,7.466429,7.510714,447610800.0,3.388733e+09
...,...,...,...,...,...,...,...,...,...
149439,2024-12-23,XOM,102.536774,106.300003,106.599998,104.919998,105.309998,12285100.0,1.305906e+09
149440,2024-12-24,XOM,102.633224,106.400002,107.190002,105.699997,106.519997,7807000.0,8.306648e+08
149441,2024-12-26,XOM,102.720039,106.489998,107.029999,105.940002,106.519997,9652400.0,1.027884e+09
149442,2024-12-27,XOM,102.710388,106.480003,107.989998,105.769997,106.300003,11943900.0,1.271787e+09


-------------------

In [21]:
#Now moving on to creating lables using forward returns

def compute_daily_returns(df):
    df = df.copy()

    df["return_1d"] = (
        df.groupby("ticker")["adj_close"]
        .pct_change()
    )

    return df

def compute_forward_returns(df, horizons=(5, 21)):
    df = df.copy()

    for h in horizons:
        df[f"fwd_return_{h}d"] = (
            df.groupby("ticker")["adj_close"]
            .shift(-h) / df["adj_close"] - 1
        )

    return df

df = compute_daily_returns(df)

df = compute_forward_returns(df)

df

,date,ticker,adj_close,close,high,low,open,volume,dollar_volume,return_1d,fwd_return_5d,fwd_return_21d
0,2010-01-04,AAPL,6.418382,7.643214,7.660714,7.585000,7.622500,493729600.0,3.773681e+09,NaN,-0.018223,-0.069062
1,2010-01-05,AAPL,6.429479,7.656429,7.699643,7.616071,7.664286,601904800.0,4.608441e+09,0.001729,-0.031066,-0.104160
2,2010-01-06,AAPL,6.327210,7.534643,7.686786,7.526786,7.656429,552160000.0,4.160329e+09,-0.015906,-0.001517,-0.073518
3,2010-01-07,AAPL,6.315512,7.520714,7.571429,7.466071,7.562500,477131200.0,3.588367e+09,-0.001849,-0.005461,-0.078165
4,2010-01-08,AAPL,6.357501,7.570714,7.571429,7.466429,7.510714,447610800.0,3.388733e+09,0.006648,-0.028540,-0.074488
...,...,...,...,...,...,...,...,...,...,...,...,...
149439,2024-12-23,XOM,102.536774,106.300003,106.599998,104.919998,105.309998,12285100.0,1.305906e+09,0.004062,NaN,NaN
149440,2024-12-24,XOM,102.633224,106.400002,107.190002,105.699997,106.519997,7807000.0,8.306648e+08,0.000941,NaN,NaN
149441,2024-12-26,XOM,102.720039,106.489998,107.029999,105.940002,106.519997,9652400.0,1.027884e+09,0.000846,NaN,NaN
149442,2024-12-27,XOM,102.710388,106.480003,107.989998,105.769997,106.300003,11943900.0,1.271787e+09,-0.000094,NaN,NaN


In [22]:
def final_cleanup(df):
    
    df = df.dropna()
    df.reset_index(drop=True, inplace=True)
    return df

df = final_cleanup(df)

In [23]:
df

,date,ticker,adj_close,close,high,low,open,volume,dollar_volume,return_1d,fwd_return_5d,fwd_return_21d
0,2010-01-05,AAPL,6.429479,7.656429,7.699643,7.616071,7.664286,601904800.0,4.608441e+09,0.001729,-0.031066,-0.104160
1,2010-01-06,AAPL,6.327210,7.534643,7.686786,7.526786,7.656429,552160000.0,4.160329e+09,-0.015906,-0.001517,-0.073518
2,2010-01-07,AAPL,6.315512,7.520714,7.571429,7.466071,7.562500,477131200.0,3.588367e+09,-0.001849,-0.005461,-0.078165
3,2010-01-08,AAPL,6.357501,7.570714,7.571429,7.466429,7.510714,447610800.0,3.388733e+09,0.006648,-0.028540,-0.074488
4,2010-01-11,AAPL,6.301419,7.503929,7.607143,7.444643,7.600000,462229600.0,3.468538e+09,-0.008821,0.023464,-0.071344
...,...,...,...,...,...,...,...,...,...,...,...,...
148559,2024-11-21,XOM,117.613434,121.930000,122.550003,120.269997,121.080002,14675400.0,1.789372e+09,0.013381,-0.032560,-0.128188
148560,2024-11-22,XOM,117.478386,121.790001,123.209999,121.639999,121.820000,13323400.0,1.622657e+09,-0.001148,-0.032351,-0.126365
148561,2024-11-25,XOM,115.722816,119.970001,121.879997,119.610001,121.430000,26580300.0,3.188839e+09,-0.014944,-0.019172,-0.112361
148562,2024-11-26,XOM,113.793625,117.970001,119.680000,117.849998,119.529999,14827300.0,1.749177e+09,-0.016671,-0.031279,-0.097398


In [24]:
df.isna().sum()

date              0
ticker            0
adj_close         0
close             0
high              0
low               0
open              0
volume            0
dollar_volume     0
return_1d         0
fwd_return_5d     0
fwd_return_21d    0
dtype: int64

In [26]:
from pathlib import Path
from config import PROCESSED_DATA_PATH

def save_processed_data(df):
    Path(PROCESSED_DATA_PATH).mkdir(parents=True, exist_ok=True)

    path = Path(PROCESSED_DATA_PATH) / "clean_panel.parquet"
    df.to_parquet(path, index=False)

    print(f"Saved processed data to {path}")

save_processed_data(df)


Saved processed data to /Users/sudhanvabharadwaj/Documents/Full_Quant_Pipeline/Data/processed/clean_panel.parquet
